# Session 7 · Multi-Feature Regression in Practice

**Machine Learning Foundations · Sanketana School of Code**

So far the model has looked at *one* habit. But a test score doesn't come from study hours alone. Today we hand the model **all five habits at once**, read what it learned, and answer the question this dataset was built to ask: **which habit matters most?**

By the end of this notebook you will be able to:

- fit a regression on **many features**, each with its own coefficient
- show that more ingredients **lower the cost**
- read a coefficient's **sign**, and explain why you can't compare raw coefficient **sizes**
- **scale** the features so the coefficients become comparable, then rank the ingredients

## Warm-up · Last session's homework

Your coach will walk through Session 6's housing bowl (about 10 minutes). You each got two costs and a bowl with its floor marked — and you noticed the bottom was *flat*: one feature left a lot of cost unexplained.

That's the itch we scratch today. What if we give the model **more ingredients**?

## Step 1 · From one ingredient to five

With one feature the model was a single slope. With five, a prediction becomes a **weighted sum of ingredients** — each habit gets its own coefficient:

```
prediction = c1·study + c2·attendance + c3·sleep + c4·screen + c5·practice + intercept
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

students = pd.read_csv("../../../datasets/anchor/student_habits.csv")

# ✏️ TODO: list the five habit features.
habits = ["study_hours_per_week", "attendance_pct", "sleep_hours_per_night",
          "screen_time_hours_per_day", "practice_sessions_per_week"]

X = students[habits]
y = students["test_score"]

model = LinearRegression().fit(X, y)
print("number of coefficients:", len(model.coef_), " (one per habit)")
print("intercept:", round(model.intercept_, 2))

## Step 2 · More ingredients, less cost

Remember **cost** from Session 6 — the average squared miss, one number for how wrong the model is. Let's compare the study-hours-only model to the all-five model.

In [ ]:
def cost(y_true, y_pred):
    """Average squared miss — the cost from Session 6."""
    return np.mean((y_true - y_pred)**2)

one = LinearRegression().fit(students[["study_hours_per_week"]], y)
cost_one = cost(y, one.predict(students[["study_hours_per_week"]]))
cost_five = cost(y, model.predict(X))

print(f"cost with study hours only: {cost_one:.1f}")
print(f"cost with all five habits:  {cost_five:.1f}")
print("More good ingredients let the line fit better, so the cost fell.")

## Step 3 · Read the coefficients

Now the door Session 4 kept shut. Each habit has a coefficient. Two things to read:

- **sign** — positive pushes the score up, negative pulls it down
- **size** — *seems* like importance… we'll test that in a moment

Here they are, sorted by size:

In [ ]:
raw = pd.DataFrame({"habit": habits, "coefficient": model.coef_})
raw = raw.sort_values("coefficient", key=lambda c: c.abs(), ascending=False)
raw.round(3)

### ✏️ First read

1. Which habit has a **negative** coefficient? What does that claim, in plain English?
2. By raw size, which habit *looks* most important? Which looks least?

*Your answers:*

1. 
2. 

## Step 4 · The trap — those sizes aren't a fair race

`sleep` has the biggest raw number and `attendance` the smallest. But look at their **units**:

- sleep is in **hours** — everyone sleeps between about 5 and 9, a narrow range, so "points per hour" is a big number.
- attendance is in **percent** — spread from 55 to 100, a wide range, so "points per percent" is a small number.

Comparing them by raw size is like racing a speed in km/h against one in miles/h by looking only at the number. To compare fairly, every feature must be in the **same** units.

In [ ]:
# Proof they're on different scales: look at each habit's spread.
students[habits].agg(["min", "max", "std"]).round(1)

## Step 5 · Level the field with scaling

Back in Session 3 you met **scaling** — putting every feature on the same footing — and we said you'd use it when it mattered. It matters now. We rescale every habit to the same "one standard step" and refit.

Watch two things: first the **cost doesn't change** (same model, same predictions), and then the coefficients come out **comparable**.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
scaled_model = LinearRegression().fit(X_scaled, y)

# 1) Did scaling change the model? Compare the costs.
print(f"cost unscaled: {cost(y, model.predict(X)):.1f}")
print(f"cost scaled:   {cost(y, scaled_model.predict(X_scaled)):.1f}")
print("→ identical. Scaling did NOT improve the model — it only rewrote the coefficients.\n")

# 2) Now the coefficients are comparable. Rank them.
scaled = pd.DataFrame({"habit": habits, "scaled_coefficient": scaled_model.coef_})
scaled = scaled.sort_values("scaled_coefficient", key=lambda c: c.abs(), ascending=False)
scaled.round(2)

**The ranking flipped.** Raw sizes said *sleep* mattered most and *attendance* least. On equal footing, **study hours dominate** and **attendance is a strong second** — sleep, which looked biggest, sits in the middle. Only after levelling the field can you honestly say which ingredient matters most.

## Step 6 · Name the winner

A picture of the scaled coefficients — length is importance, direction is sign.

In [ ]:
plot_df = scaled.sort_values("scaled_coefficient")
colors = ["tomato" if v < 0 else "seagreen" for v in plot_df["scaled_coefficient"]]

plt.figure(figsize=(7, 4))
plt.barh(plot_df["habit"], plot_df["scaled_coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("scaled coefficient  (importance, with direction)")
plt.title("Which ingredient matters most for a test score?")
plt.tight_layout(); plt.show()

### ✏️ Write the insight

1. Using the **scaled** table, finish: "In this data, the habit most associated with a higher test score is ____, and the one that most pulls scores down is ____."
2. Why must we say *"most associated with"* rather than *"studying more will raise your score by this much"*? (Think about what the model actually found.)

*Your answers:*

1. 
2. 

### ✏️ Stretch — does *any* extra column lower the cost?

Add a column of pure random noise as a "feature" and refit. Watch what the cost does.

In [ ]:
rng = np.random.default_rng(0)
X_junk = X.copy()
X_junk["random_noise"] = rng.normal(size=len(X_junk))

junk_model = LinearRegression().fit(X_junk, y)
print(f"cost with 5 real habits:        {cost(y, model.predict(X)):.2f}")
print(f"cost after adding pure noise:   {cost(y, junk_model.predict(X_junk)):.2f}")
print("The cost drops even for a useless column! Lower training cost is NOT")
print("automatically a better model — that trap gets its own session (16).")

## What we learned

✏️ Three quick reflections — one line each:

1. In a multi-feature model, each feature has its own ____.
2. Why you can't rank importance from raw coefficient sizes:
3. The habit that matters most for a test score, in this data:

---

**You opened the whole box.** A multi-feature regression is a weighted sum of ingredients; more ingredients lowered the cost; and — once scaled onto equal footing — the coefficients rank which ingredient matters most.

**Next (Session 8):** the cost fell from 44 to 26… but *is 26 good?* We've been comparing with cost; next session we report a model's quality in named numbers a parent would understand — MAE, RMSE, R² — and finally test on data the model has never seen.